# 04 — Data storytelling para el cliente

Narrativa ejecutiva del experimento de emails con nudges.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import GROUP_LABELS, load_data

df = load_data(ROOT / 'data' / 'datos_prueba_tecnica.csv')

## 1. El reto del banco

In [ ]:
summary = df.groupby('grupo', observed=True)[['or', 'ctor']].mean()
summary.index = summary.index.map(GROUP_LABELS)
ctrl_or, ctrl_ctor = summary.loc[GROUP_LABELS['ctrl']]
best = summary.loc[GROUP_LABELS['trat2']]
print(f'Control: {ctrl_or:.1%} apertura, {ctrl_ctor:.1%} clics')
print(f'Trat2:   {best["or"]:.1%} apertura, {best["ctor"]:.1%} clics')

## 2. Resultado del experimento

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = df.groupby('grupo', observed=True)[['or', 'ctor']].mean().reset_index()
plot_data['grupo'] = plot_data['grupo'].map(GROUP_LABELS)
plot_melt = plot_data.melt(id_vars='grupo', var_name='metric', value_name='rate')
sns.barplot(data=plot_melt, x='grupo', y='rate', hue='metric', ax=ax)
ax.set_ylabel('Proporción')
ax.set_title('Open rate y click rate por variante de email')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 3. Impacto estimado

In [ ]:
n_clients = 500_000
lift_ctor = best['ctor'] - ctrl_ctor
extra_clicks = int(lift_ctor * n_clients)
print(f'Si escalamos trat2 a {n_clients:,} clientes:')
print(f'  ~{extra_clicks:,} clics adicionales vs control ({lift_ctor:.1%} lift absoluto)')

## 4. Recomendación

In [ ]:
recommendation = pd.DataFrame({
    'Acción': [
        'Desplegar email Tratamiento 2 como variante principal',
        'Priorizar segmentos con mayor CATE (usuarios app, edad media)',
        'Monitorizar A/B continuo post-lanzamiento',
    ],
    'Impacto esperado': [
        f'+{lift_ctor:.1%} en click rate vs control',
        'Optimización adicional vía personalización',
        'Detección temprana de fatiga del nudge',
    ],
})
recommendation

---

**Exportar:** ver [`docs/04_DATA_STORYTELLING.md`](../docs/04_DATA_STORYTELLING.md)